In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.metrics import precision_recall_curve

In [5]:
df = pd.read_csv("processed_fraud_data.csv")

In [6]:
df = df.drop("AmountBin", axis=1)

In [5]:
#df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 283726 entries, 0 to 283725
Data columns (total 35 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Time          283726 non-null  float64
 1   V1            283726 non-null  float64
 2   V2            283726 non-null  float64
 3   V3            283726 non-null  float64
 4   V4            283726 non-null  float64
 5   V5            283726 non-null  float64
 6   V6            283726 non-null  float64
 7   V7            283726 non-null  float64
 8   V8            283726 non-null  float64
 9   V9            283726 non-null  float64
 10  V10           283726 non-null  float64
 11  V11           283726 non-null  float64
 12  V12           283726 non-null  float64
 13  V13           283726 non-null  float64
 14  V14           283726 non-null  float64
 15  V15           283726 non-null  float64
 16  V16           283726 non-null  float64
 17  V17           283726 non-null  float64
 18  V18 

In [7]:
X = df.drop("Class", axis=1)
y = df["Class"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,stratify=y,random_state=42)

In [9]:
lr = LogisticRegression(max_iter=1000,class_weight="balanced")

In [10]:
lr.fit(X_train, y_train)

D:\C_disk\CONDA\envs\world1\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [11]:
y_prob_lr = lr.predict_proba(X_test)[:, 1]
y_pred_lr = (y_prob_lr >= 0.175).astype(int)  

In [12]:
print(classification_report(y_test, y_pred_lr))
print(confusion_matrix(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))

              precision    recall  f1-score   support

           0       1.00      0.87      0.93     56651
           1       0.01      0.92      0.02        95

    accuracy                           0.87     56746
   macro avg       0.51      0.89      0.48     56746
weighted avg       1.00      0.87      0.93     56746

[[49180  7471]
 [    8    87]]
ROC-AUC: 0.9647538158382488


In [15]:
precision_lr, recall_lr, thresholds_lr = precision_recall_curve(y_test, y_prob_lr)

f1_scores_lr = 2 * (precision_lr * recall_lr) / (precision_lr + recall_lr + 1e-10)

best_index_lr = np.argmax(f1_scores_lr)
best_threshold_lr = thresholds_lr[best_index_lr]

print("Best Threshold (LR):", best_threshold_lr)
print("Best F1:", f1_scores_lr[best_index_lr])
print("Precision:", precision_lr[best_index_lr])
print("Recall:", recall_lr[best_index_lr])

Best Threshold (LR): 0.9999724235973196
Best F1: 0.7999999999501544
Precision: 0.8470588235294118
Recall: 0.7578947368421053


In [1]:
#pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.8/101.7 MB 6.7 MB/s eta 0:00:16
    --------------------------------------- 1.8/101.7 MB 4.6 MB/s eta 0:00:22
   - -------------------------------------- 2.6/101.7 MB 3.9 MB/s eta 0:00:26
   - -------------------------------------- 3.7/101.7 MB 4.2 MB/s eta 0:00:24
   - -------------------------------------- 4.5/101.7 MB 4.1 MB/s eta 0:00:24
   -- ------------------------------------- 5.2/101.7 MB 4.1 MB/s eta 0:00:24
   -- ------------------------------------- 6.0/101.7 MB 4.1 MB/s eta 0:00:24
   -- ------------------------------------- 6.8/101.7 MB 4.0 MB/s eta 0:00:24
   --- ------------------------------------ 7.9/101.7 MB 4.0 MB/s eta 0:00:24
   --- ------------------------------------ 8.7/101.7 MB 4.0 MB/s eta 0:00:24
   --- ------------------------------------ 9.4/101.7 MB 4.0 MB/s eta 0:00:24
   ---- ----------------------------------- 10.2/101.7 MB 4.0 MB/s eta 

In [2]:
from xgboost import XGBClassifier

In [9]:
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])

xgb = XGBClassifier(n_estimators=300,max_depth=4,learning_rate=0.1,scale_pos_weight=scale_pos_weight,random_state=42,n_jobs=-1,eval_metric="logloss")

In [10]:
xgb.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [11]:
y_prob_xgb = xgb.predict_proba(X_test)[:,1]

In [12]:
precision_xgb, recall_xgb, thresholds_xgb = precision_recall_curve(y_test, y_prob_xgb)
f1_xgb = 2 * (precision_xgb * recall_xgb) / (precision_xgb + recall_xgb + 1e-10)

best_index_xgb = np.argmax(f1_xgb)
best_threshold_xgb = thresholds_xgb[best_index_xgb]

In [13]:
print("Best Threshold (XGB):", best_threshold_xgb)
print("Best F1:", f1_xgb[best_index_xgb])
print("Precision:", precision_xgb[best_index_xgb])
print("Recall:", recall_xgb[best_index_xgb])

Best Threshold (XGB): 0.9443284
Best F1: 0.8571428570937145
Precision: 0.9863013698630136
Recall: 0.7578947368421053
